In [ ]:
import pandas as pd
import re
import ast
import plotly.express as px
from cleantext import clean

In [ ]:
jsonl_file = r"train.jsonl"
ds = pd.read_json(jsonl_file, lines=True)
ds.to_csv(r"train.csv", index=False)


In [ ]:
df = pd.read_csv(r"train.csv")

In [ ]:
df.info()

In [ ]:
print(df.head())

In [ ]:
df.columns.tolist()

In [ ]:
fig = px.histogram(df, x='labels')
fig.update_layout(xaxis_tickvals=[])
fig.show()

Processing

In [ ]:
unique_labels = set()
for labels_list in df['emotions_used_to_generate_context']:
    for label in eval(labels_list):
        unique_labels.add(label)
for label in unique_labels:
    print(label)

In [ ]:
excluded = {
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
}

unique_labels = set()
for labels_list in df['emotions_used_to_generate_context']:
    try:
        labels = ast.literal_eval(labels_list)
    except Exception:
        # skip malformed entries
        continue
    unique_labels.update(labels)

remaining = sorted(unique_labels - excluded)

if remaining:
    print("Labels not in the excluded set:")
    for lbl in remaining:
        print(lbl)
else:
    print("No labels remain after excluding the provided set.")

In [ ]:
non_matched_emotions = ["actor", "addison", "agnes", "albert", "alice", "ambition", "amused", "amusedly", "anderson", "angalo", "angeline", "angelo", "angie", "angrily", 
                        "angry", "angua", "anguish", "angus", "anita", "anne", "annoyed", "anticipation", "anxiety", "anxious", "anxiously", "approvingly", "arabel",
                          "arrhenius", "arrogance", "ashlee", "awe", "ba", "betty", "bitterly", "bitterness", "boredom", "boris", "calmly", "caution", 
                          "celia", "character", "charlene", "charles", "charlotte", "compassion", "concern", "confidence", "confused", "confusedly", "contempt", 
                          "contemptuously", "contentment", "country", "curious", "curiously", "damien", "deception", "delight", "delusion", "denise", "desiring", 
                          "desiringly", "despair", "desperately", "determination", "determined", "direction", "disappointed", "disappointedly", "disapproving", 
                          "disapprovingly", "disbelief", "disdain", "disdainfully", "disgusted", "distrust", "doubt", "dumpty", "eddorian", "egbert", "egwene", 
                          "elaine", "elanora", "elrohir", "elton", "emmy", "emotion", "envy", "ercles", "ernie", "ethan", "exasperation", "excited", "excitedly", 
                          "fascination", "fearful", "fearfully", "formality", "freddy", "freyr", "frodo", "frustrated", "frustration", "georgette", "glee", "gloating",
                            "grateful", "gratefully", "greed", "grudge", "grudgingly", "guilt", "happily", "hare", "hate", "hatred", "helena", "helplessness", "henia", 
                            "hope", "hopefully", "hopefulness", "horror", "hurt", "ida", "ignorance", "igor", "indifference", "indignation", "irene", "irritation", 
                            "irwin", "jake", "jealous", "jealously", "jealousy", "joyful", "joyfully", "julia", "laughing", "longing", "longingly", "lorena", 
                            "loving", "lovingly", "lustfully", "martha", "melanie", "melinda", "merry", "michelle", "mike", "mitzi", "monica", "nan", "nervous", 
                            "nervously", "nostalgia", "optimistic", "optimistically", "orion", "outrage", "pain", "panic", "paranoia", "pity", "plot", "protectively", 
                            "proud", "proudly", "quina", "rage", "rainie", "regret", "resentment", "resignation", "revenge", "roony", "ruth", "sad", "sadism", 
                            "sadistically", "sadly", "satisfaction", "scared", "scornfully", "shame", "shock", "skepticism", "smiling", "smirking", "sorrow", 
                            "surprised", "surprisedly", "suspicion", "suspiciously", "sympathy", "terror", "thinking", "thought", "thread", "triumph", "triumphantly", 
                            "whispering", "willie", "wonder", "worried", "worry"
]

def contains_non_matched(emotion_list_str):
    if pd.isna(emotion_list_str):
        return False
    try:
        emotion_list = ast.literal_eval(emotion_list_str)
        if not isinstance(emotion_list, list):
            return False
        return any(e in non_matched_emotions for e in emotion_list)
    except (ValueError, SyntaxError):
        return False

filtered_df = df[~df["emotions_used_to_generate_context"].apply(contains_non_matched)]
print(f"Kept {len(filtered_df)} rows after discarding rows with non-matched emotions.")


In [ ]:
filtered_df.columns.tolist()

In [ ]:
df1 = filtered_df.copy()
df1.drop(
    ['plot_id','primary_emotion','all_emotions','all_emotions_mapped',
     'raw_emotion_explication','expressiveness','emotions_used_to_generate_context'],
    axis=1,
    inplace=True
)
df1.head()

In [ ]:
df1.rename(columns={'utterance': 'text'}, inplace=True)
df1.head()

In [ ]:
df1.info()

In [ ]:
def clean_text(text):
    # 1. Use cleantext to fix Unicode + transliterate
    text = clean(text,
                 fix_unicode=True,
                 to_ascii=True,
                 lower=False,
                 no_emoji=True,
                 no_urls=True,
                 no_currency_symbols=False)

    # 2. Remove stray currency symbols NOT followed by a number
    text = re.sub(r'([$€£₹])(?!\d)', '', text)

    # 3. Remove unwanted punctuation excluding . , ! ? $ % &
    text = re.sub(r'[{}\[\]<>@#*`~/\\\^_|+=\-—…;:\(\)]', '', text)

    # 5. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df1['text'] = df1['text'].apply(clean_text)
df1.head()


In [ ]:
print(df1['text'].iloc[49])

In [ ]:
print(type(df1['labels'][6]))

In [ ]:
df1['labels'] = df1['labels'].apply(ast.literal_eval)
print(type(df1['labels'][5]))

In [ ]:
df1.head(10)

In [ ]:
one_hot = pd.get_dummies(df1['labels'].explode()).groupby(level=0).max().astype(int)
df2 = pd.concat([df1, one_hot], axis=1)
df2.head()

In [ ]:
# Get the integer column names from index 2 to 29 
int_cols = [str(i) for i in sorted([int(col) for col in df2.columns[2:30]])]
# Reorder columns
df3 = df2[df2.columns[:2].tolist() + int_cols]
df3.head()

In [ ]:
print(df3.columns)

#### Balancing the dataset

In [ ]:
count_label= df3['labels'].value_counts()
for label, count in count_label.items():
    print(f"{label}: {count}")


In [ ]:
common_labels = count_label[count_label >= 100].index
df_label = df3[df3['labels'].isin(common_labels)]
df_label.info()

In [ ]:
# quick diagnostics
print("Total rows:", len(df3))
print("Selected rows:", len(df_label))
print("Sample common labels:", list(common_labels)[:5])

In [ ]:
label_counts = df_label['labels'].value_counts().sort_index()

In [ ]:
fig = px.bar(label_counts, x=label_counts.index.astype(str), y=label_counts.values,
             labels={'x': 'Label', 'y': 'Count'},
             title='Distribution of Individual Labels After Filtering & Applying Lower Bound'
             )
fig.update_layout(xaxis_tickvals=[])
fig.show()


In [ ]:
fig = px.bar(label_counts, x=label_counts.index.astype(str), y=label_counts.values,
             labels={'x': 'Label', 'y': 'Count'},
             title='Distribution of Individual Labels After Filtering & Applying Lower Bound',
             log_y=True)
fig.update_layout(xaxis_tickvals=[])
fig.show()


In [ ]:
df_label.info()

In [ ]:
thresh = 531

df_balanced = (
    df_label.groupby(df_label['labels'].apply(tuple), group_keys=False)
            .apply(lambda x: x.sample(n=min(len(x), thresh), random_state=42))
            .reset_index(drop=True)
)

print(f"Rows after balancing labels: {len(df_balanced)}")
print(df_balanced['labels'].value_counts())


In [ ]:
df_balanced.info()

In [ ]:
label_counts_new = df_balanced['labels'].value_counts().sort_index()

In [ ]:
fig = px.bar(label_counts_new, x=label_counts_new.index.astype(str), y=label_counts_new.values,
             labels={'x': 'Label', 'y': 'Count'},
             title='Distribution of Individual Labels After Balancing',
             log_y=True
             )
fig.update_layout(xaxis_tickvals=[])
fig.show()


In [ ]:
df_balanced.head(5)

In [ ]:
df_balanced.drop(columns=['labels'], inplace=True)
df_balanced.head()

In [ ]:
df_final = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.head()

In [ ]:
df_final.to_csv(r"EmoPillar-Train.csv", index=False)